# Fooocus 2.5.6 - Colab Edition

1. Comprueba que el entorno tenga GPU: **Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU**.
2. Ejecuta la celda de abajo.
3. Abre la **URL PUBLICA** que aparece en un recuadro de `=====` a los pocos segundos.

El tunel por defecto es **cloudflared**: `gradio.live` pierde paquetes en el WebSocket y
deja la UI en `Loading models...` aunque la imagen ya este generada en `outputs/`.

La primera vez descarga unos 9 GB de modelos (checkpoint + inpaint + expansion), con
aria2c a 16 conexiones. Activa `CACHEAR_MODELOS_EN_DRIVE` para que pase una sola vez.

In [ ]:
# @title Arrancar Fooocus
BRANCH = 'colab-2.5.6'  # @param {type:"string"}
TUNEL = 'cloudflare'  # @param ["cloudflare", "gradio"]
CACHEAR_MODELOS_EN_DRIVE = False  # @param {type:"boolean"}
EXTRAS_OPCIONALES = False  # @param {type:"boolean"}
ARGUMENTOS_EXTRA = '--always-high-vram'  # @param {type:"string"}

import os
import re
import shlex
import shutil
import subprocess
import sys

REPO = 'https://github.com/deleonramiro085/Fooocus.git'
WORKDIR = '/content/Fooocus'
PUERTO = 7865
DRIVE_CACHE = '/content/drive/MyDrive/Fooocus/models'
SHARED_SUBDIRS = ['checkpoints', 'loras', 'inpaint', 'controlnet', 'clip_vision',
                  'upscale_models', 'vae', 'vae_approx', 'sam', 'safety_checker']
CLOUDFLARED_DEB = ('https://github.com/cloudflare/cloudflared/releases/latest/download/'
                   'cloudflared-linux-amd64.deb')

gpu = ''
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    pass
print('GPU:', gpu or 'NO DETECTADA')
if not gpu:
    print('Sin GPU no hay nada que hacer: Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU.')

if not os.path.isdir(os.path.join(WORKDIR, '.git')):
    shutil.rmtree(WORKDIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, WORKDIR], check=True)
os.chdir(WORKDIR)

# aria2c: Hugging Face limita el descargador de una sola conexion a ~70 kB/s en Colab.
# modules/model_loader.py lo detecta y lo usa solo (16 conexiones, reanudable).
if shutil.which('aria2c') is None:
    print('Instalando aria2 (descargas multi-hilo)...')
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'aria2'], check=False)

if CACHEAR_MODELOS_EN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    for sub in SHARED_SUBDIRS:
        target = os.path.join(DRIVE_CACHE, sub)
        os.makedirs(target, exist_ok=True)
        local = os.path.join(WORKDIR, 'models', sub)
        if os.path.islink(local):
            continue
        shutil.rmtree(local, ignore_errors=True)
        os.symlink(target, local)
    print('Modelos cacheados en', DRIVE_CACHE)

tunel_url = None
if TUNEL == 'cloudflare':
    if shutil.which('cloudflared') is None:
        deb = '/content/cloudflared-linux-amd64.deb'
        subprocess.run(['wget', '-q', '-nc', '-O', deb, CLOUDFLARED_DEB], check=False)
        subprocess.run(['dpkg', '-i', deb], check=False,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

if TUNEL == 'cloudflare' and shutil.which('cloudflared') is not None:
    # El tunel se levanta antes que el servidor a proposito: cloudflared responde 502
    # hasta que Fooocus escucha, y asi la URL ya esta impresa cuando termina el arranque.
    proceso = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PUERTO}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(120):
        linea = proceso.stdout.readline()
        if not linea:
            break
        encontrado = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', linea)
        if encontrado:
            tunel_url = encontrado.group(0)
            break
    if tunel_url:
        print('\n' + '=' * 72)
        print('URL PUBLICA DE LA INTERFAZ:', tunel_url)
        print('(tarda 3-8 minutos en responder: esta descargando modelos)')
        print('=' * 72 + '\n')
    else:
        print('No se pudo abrir el tunel de Cloudflare. Se cae a --share de gradio.')
        proceso.kill()

cmd = [sys.executable, '-u', 'entry_with_update.py', '--skip-update',
       '--preset', 'default', '--disable-analytics', '--port', str(PUERTO)]
if tunel_url:
    cmd += ['--listen', '127.0.0.1']
else:
    cmd.append('--share')
if EXTRAS_OPCIONALES:
    cmd.append('--install-optional')
cmd += shlex.split(ARGUMENTOS_EXTRA)
print(' '.join(cmd))
subprocess.run(cmd, check=False)


In [ ]:
# @title Diagnostico (solo si algo falla)
import importlib.metadata as md
import platform
import shutil

print('python', platform.python_version())
try:
    import torch
    print('torch', torch.__version__, '| cuda', torch.version.cuda)
    print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU')
except Exception as e:
    print('torch no importable:', e)
print('aria2c', shutil.which('aria2c') or 'no instalado')
print('cloudflared', shutil.which('cloudflared') or 'no instalado')
for p in ('gradio', 'gradio_client', 'numpy', 'transformers', 'tokenizers', 'huggingface_hub',
          'pydantic', 'fastapi', 'starlette', 'websockets', 'python-multipart',
          'onnxruntime', 'pygit2', 'pillow'):
    try:
        print(p, md.version(p))
    except Exception:
        print(p, 'no instalado')


## Si algo va mal

- **La URL da 502 o `Bad Gateway`**: normal al principio, el primer arranque son 3-8 minutos.
- **La UI se queda en `Loading models...`**: es el WebSocket de `gradio.live`. Usa `TUNEL = cloudflare`.
- **`CUDA out of memory`**: borra `--always-high-vram` de `ARGUMENTOS_EXTRA`.
- **Descarga lentisima (kB/s)**: comprueba en la celda de diagnostico que `aria2c` esta instalado.
- **Empezar de cero**: borra la carpeta `/content/Fooocus` y vuelve a ejecutar.
- **Mascara automatica / quitar fondo**: activa `EXTRAS_OPCIONALES` (rembg, SAM, GroundingDINO).